In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import trajectory_data
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dino_model import DINOFeaturePresence, DINOWithMIL
from nocode_robot_programming.state_decision.state_decider_alt import DINOProtoPresence
# from nocode_robot_programming.state_decision.dino_model_v2 import DINOFeaturePresence
# from nocode_robot_programming.state_decision.dino_model_v3 import DINOFeaturePresence
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.utils import Filename
from gesture_detector.utils import pretty_confusion_matrix
import torch
import numpy as np

from trajectory_data.skill_visualizer import play_video
from nocode_robot_programming.state_decision.utils import add_tag
from nocode_robot_programming.state_decision.utils import visualize_accuracies
from nocode_robot_programming.state_decision.dataloader import ImageDatasetView, saved_img_processing
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda

seed = 49
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

from nocode_robot_programming.state_decision.dataset_d1 import d1_peg_pick, d2_peg_pick, d1_probe, d2_probe, dupl, get_dataset_view
datafileloader = TrajectoryDataset(trajectory_data.package_path)
datasets = d1_peg_pick(datafileloader)

# Some images from the dataset

In [ ]:
d_train, d_test, d_text = datasets[0]
# d_train.showcase(fps=20)
# d_train.showcase_captions(fps=20)
d_train.play_video(fps=2)
# d_test.play_video(fps=10)

# Checkpoint 2025-10-16

- Train and test accuracy achieved 100%

In [ ]:
seed = 44
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

stats1, stats2 = [], []

for decider in [
        DINOProtoPresence(),
        # DINOFeaturePresence(percentile_keep=0.0, dino_variant="dinov2_vitg14"),
        DINOFeaturePresence(percentile_keep=0.0),
        DINOWithMIL(),
        # DINOStateDecider(dino_variant = "dinov2_vitl14", percent_keep=0),
        # StateDeciderSIFT(),
        # AEGP(),
    ]:
    s1, s2 = [], []
    for task_dataset_gen in [d1_peg_pick, d2_peg_pick, d1_probe, d2_probe]:
        
        for d_train, d_test, d_text in task_dataset_gen(datafileloader):
    
            print("-" * 30)
            print(f"Model: {decider.__class__.__name__}, {d_text}")
            
            decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function

            y_pred = decider.predict_many(d_train.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False); ipt.save()
            s1.append((np.array(d_train.y_names) == np.array(y_pred)).mean())

            y_pred = decider.predict_many(d_test.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
            s2.append((np.array(d_test.y_names) == np.array(y_pred)).mean())
            
            ipt.show()

    stats1.append(s1)
    stats2.append(s2)


demo_models = ["DINOFeaturePresence", "DINOFeaturePresence", "DINOWithMIL"]
demo_tasks = ["d1_peg_pick_mix1", "d1_peg_pick_mix2", "d1_peg_pick_mix3", 
              "d2_peg_pick_mix1", "d2_peg_pick_mix2", "d2_peg_pick_mix3", 
              "d1_probe_mix1", "d1_probe_mix2", "d1_probe_mix3", 
              "d2_probe_mix1", "d2_probe_mix2", "d2_probe_mix3"]

visualize_accuracies(100 * np.array(stats1), 100 * np.array(stats2), demo_models, demo_tasks, out_dir="plot")

#### Play all frames that are wrongly predicted

- you can modify speed with `fps`
- you can modify window size with `scale`

In [ ]:
play_video(
    d_test.X[np.array(d_test.y_names) != np.array(decider.predict_many(d_test.X))].cpu().numpy() * 256,
    fps=2, window_name="win", backend="cv2", scale=10.0)

In [ ]:
play_video(
    d_train.X[np.array(d_train.y_names) != np.array(decider.predict_many(d_train.X))].cpu().numpy() * 256,
    fps=2, window_name="win", backend="cv2", scale=10.0)

# TODO:

- Good training depends on good starting point, it doesn't have to converge

### See if accuracy varies further from the branch timestep

In [ ]:
plt.hist(d_test.Xt[np.array(d_test.y_names) == np.array(decider.predict_many(d_test.X))].cpu().numpy())